# JAXTPC Quickstart

The smallest end-to-end example: build a wire detector, generate a synthetic
multi-track event, run the simulation, and plot the readout. No external data
needed. To simulate real data, swap `make_synthetic_event(...)` for
`load_event(path, cfg, event_idx=...)`.


In [ ]:
# Resolve the repo root so `import tools` and config/ paths work from any folder.
import os, sys
_d = os.path.abspath(os.getcwd())
while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, 'config')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)


In [ ]:
import jax, numpy as np, matplotlib.pyplot as plt
from tools.simulation import DetectorSimulator
from tools.geometry import generate_detector
from tools.loader import build_deposit_data
from tools.output import to_sparse
from tools.visualization import visualize_wire_signals

# Build a dual-TPC wire detector
detector = generate_detector('config/cubic_wireplane_config.yaml')
sim = DetectorSimulator(detector, include_track_hits=True, include_digitize=True,
                        total_pad=50_000, response_chunk_size=10_000)
cfg = sim.config


## Generate a synthetic event
A few straight MIP-like tracks crossing the detector.


In [ ]:
def make_synthetic_event(seed=0, n_tracks=6, step_cm=0.4):
    rng = np.random.RandomState(seed)
    P, DE, DX, TID = [], [], [], []
    for t in range(n_tracks):
        start = rng.uniform(-180, 180, 3); d = rng.normal(size=3); d /= np.linalg.norm(d)
        n = int(rng.uniform(80, 250) / step_cm); s = np.arange(n) * step_cm
        pts = np.clip(start[None, :] + s[:, None] * d[None, :], -215.9, 215.9)
        P.append(pts); DE.append(np.full(n, rng.uniform(1.8, 2.6) * step_cm, np.float32))
        DX.append(np.full(n, step_cm, np.float32)); TID.append(np.full(n, t, np.int32))
    return (np.concatenate(P) * 10).astype(np.float32), np.concatenate(DE), np.concatenate(DX), np.concatenate(TID)

pos_mm, de, dx, track_ids = make_synthetic_event()
deposits = build_deposit_data(pos_mm, de, dx, cfg, track_ids=track_ids)
print('deposits:', sum(int(v.n_actual) for v in deposits.volumes))


## Run the simulation


In [ ]:
sim.warm_up()
signals, track_hits_raw, deposits = sim.process_event(deposits, key=jax.random.PRNGKey(42))
sparse = to_sparse(signals, cfg, threshold_adc=1200 / cfg.electrons_per_adc)
print('sparse signal entries:', sum(len(d['values']) for d in sparse.values()))


## Plot the wire readout


In [ ]:
fig = visualize_wire_signals(sparse, cfg, threshold_enc=1200, gamma=0.2, sparse=True)
plt.show()


## Next steps
- `wire_simulation.ipynb` — the full walkthrough (track-hit truth, per-track labels)
- `../readout/pixel_simulation.ipynb` — pixel readout
- `../physics/` — the response chain stage by stage
- `../gradients/` — the differentiable path
